In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from crc_grn import run_real_pipeline, audit_project_inputs

REAL_DATA_ROOT = ROOT / 'data' / 'main_visiumhd'
REAL_PRIOR_ROOT = ROOT / 'resources' / 'grn_prior'
SC_REFERENCE_ROOT = ROOT / 'data' / 'sc_reference'
ORTHOGONAL_XENIUM_ROOT = ROOT / 'data' / 'orthogonal_xenium_gse280314'
EXTERNAL_VALIDATION_ROOT = ROOT / 'data' / 'external_validation_gse226997'
EXTENSION_ROOT = ROOT / 'data' / 'extension_gse267401'
CONFIG_PATH = ROOT / 'config' / 'samples.yaml'
MECHANISM_PANEL_PATH = ROOT / 'config' / 'mechanism_panel.yaml'
RESULTS_OUTDIR = ROOT / 'results' / 'real_run'

COLLECTRI_MIN_ROWS = 1000
TRRUST_MIN_ROWS = 100


def validate_prior_file(path: Path, min_rows: int, required: list[str]):
    if not path.exists():
        return {'file': path.name, 'ok': False, 'reason': 'missing', 'n_rows': 0}
    df = pd.read_csv(path, sep='	', low_memory=False)
    missing = [c for c in required if c not in df.columns]
    if missing:
        return {'file': path.name, 'ok': False, 'reason': f'missing columns: {missing}', 'n_rows': int(len(df))}
    if len(df) < min_rows:
        return {'file': path.name, 'ok': False, 'reason': f'too few rows: {len(df)}', 'n_rows': int(len(df))}
    if 'references' in df.columns and df['references'].astype(str).str.contains('demo_ref', case=False, na=False).all():
        return {'file': path.name, 'ok': False, 'reason': 'looks like demo prior', 'n_rows': int(len(df))}
    return {'file': path.name, 'ok': True, 'reason': '', 'n_rows': int(len(df))}


In [2]:
audit = audit_project_inputs(ROOT, CONFIG_PATH)
main_df = audit['main_visiumhd']
prior_df = pd.DataFrame([
    validate_prior_file(REAL_PRIOR_ROOT / 'collectri_human.tsv', COLLECTRI_MIN_ROWS, ['source_genesymbol', 'target_genesymbol', 'consensus_stimulation', 'consensus_inhibition', 'references', 'sources']),
    validate_prior_file(REAL_PRIOR_ROOT / 'trrust_human.tsv', TRRUST_MIN_ROWS, ['source_genesymbol', 'target_genesymbol', 'references', 'sources']),
])

display(main_df)
display(prior_df)

if main_df.empty or not main_df.groupby('sample')['exists'].sum().ge(2).all():
    raise RuntimeError('真实 discovery 输入不完整：请确认 5 个样本都至少有 filtered_feature_bc_matrix.h5 和 tissue_positions.parquet(.gz)。')
if not prior_df['ok'].all():
    raise RuntimeError('真实 prior 未通过校验：请先运行 00_download_grn_prior.ipynb 或手动放入完整官方 prior。')

print('[ok] audit passed; ready for real discovery.')


,sample,gsm,basename,exists,path
0,P1CRC,GSM8594567,filtered_feature_bc_matrix.h5,True,C:\Users\Simon\Downloads\crc_grn_optimized_v3\...
1,P1CRC,GSM8594567,Metadata.parquet.gz,True,C:\Users\Simon\Downloads\crc_grn_optimized_v3\...
2,P1CRC,GSM8594567,scalefactors_json.json.gz,True,C:\Users\Simon\Downloads\crc_grn_optimized_v3\...
3,P1CRC,GSM8594567,tissue_lowres_image.png.gz,True,C:\Users\Simon\Downloads\crc_grn_optimized_v3\...
4,P1CRC,GSM8594567,tissue_positions.parquet.gz,True,C:\Users\Simon\Downloads\crc_grn_optimized_v3\...
5,P2CRC,GSM8594568,filtered_feature_bc_matrix.h5,True,C:\Users\Simon\Downloads\crc_grn_optimized_v3\...
6,P2CRC,GSM8594568,Metadata.parquet.gz,True,C:\Users\Simon\Downloads\crc_grn_optimized_v3\...
7,P2CRC,GSM8594568,scalefactors_json.json.gz,True,C:\Users\Simon\Downloads\crc_grn_optimized_v3\...
8,P2CRC,GSM8594568,tissue_lowres_image.png.gz,True,C:\Users\Simon\Downloads\crc_grn_optimized_v3\...
9,P2CRC,GSM8594568,tissue_positions.parquet.gz,True,C:\Users\Simon\Downloads\crc_grn_optimized_v3\...


,file,ok,reason,n_rows
0,collectri_human.tsv,True,,42990
1,trrust_human.tsv,True,,8292


[ok] audit passed; ready for real discovery.


In [3]:
results = run_real_pipeline(
    data_root=REAL_DATA_ROOT,
    config_path=CONFIG_PATH,
    prior_root=REAL_PRIOR_ROOT,
    outdir=RESULTS_OUTDIR,
    fallback_prior_root=None,
    sc_reference_root=SC_REFERENCE_ROOT if SC_REFERENCE_ROOT.exists() else None,
    orthogonal_xenium_root=ORTHOGONAL_XENIUM_ROOT if ORTHOGONAL_XENIUM_ROOT.exists() else None,
    external_validation_root=EXTERNAL_VALIDATION_ROOT if EXTERNAL_VALIDATION_ROOT.exists() else None,
    extension_root=EXTENSION_ROOT if EXTENSION_ROOT.exists() else None,
    mechanism_panel_path=MECHANISM_PANEL_PATH if MECHANISM_PANEL_PATH.exists() else None,
    min_max_pct=0.03,
    min_abs_log2fc=0.15,
    top_n_extra_var=1200,
    max_spots_per_sample=1200,
    top_n_edges=200,
    top_targets_per_driver=15,
    n_niches=6,
    min_program_edges=3,
    min_edge_qvalue=0.25,
    min_edge_sample_support=3,
)

print('run mode:', results['mode'])
print('n_spots:', results['n_spots'])
print('n_genes:', results['n_genes'])
print('n_candidate_genes:', len(results['candidate_genes']))
print('n_candidate_edges:', len(results['candidate_edges']))
print('n_samples_ok:', results.get('n_samples_ok'))
print('n_samples_failed:', results.get('n_samples_failed'))
if 'qc_summary' in results:
    print('qc_summary:', results['qc_summary'])
if 'sample_logs' in results:
    display(pd.DataFrame(results['sample_logs']))


run mode: real
n_spots: 6000
n_genes: 722
n_candidate_genes: 2033
n_candidate_edges: 2387
n_samples_ok: 5
n_samples_failed: 0
qc_summary: {'candidate_genes_requested': 2033, 'candidate_edges_after_prior_filter': 2387, 'genes_after_qc': 722, 'self_loops_removed': True, 'samplewise_aggregation': True, 'n_significant_edges': 193, 'n_ccc_edges': 15768, 'n_ccc_signatures': 127, 'n_driver_ccc_associations': 12827, 'driver_program_min_edges': 3}


,sample,ok,n_spots,n_genes,error
0,P1CRC,True,1200,1935,
1,P2CRC,True,1200,1935,
2,P5CRC,True,1200,1935,
3,P3NAT,True,1200,1935,
4,P5NAT,True,1200,1935,


In [4]:
display(results['driver_ranking'].head(20))
display(results['driver_programs'].head(20))
display(results['top_edges'].head(20))
if isinstance(results.get('mechanism_hits'), pd.DataFrame) and not results['mechanism_hits'].empty:
    display(results['mechanism_hits'].head(20))
if isinstance(results.get('niche_specificity'), pd.DataFrame) and not results['niche_specificity'].empty:
    display(results['niche_specificity'].head(20))
if isinstance(results.get('gene_qc'), pd.DataFrame) and not results['gene_qc'].empty:
    display(results['gene_qc'].sort_values(['keep_after_qc', 'variance', 'nonzero_fraction'], ascending=[False, False, False]).head(20))
print('results saved to:', RESULTS_OUTDIR)


,driver,n_edges,mean_score,max_score,mean_internal,mean_external,mean_internal_support,mean_external_support,n_significant_edges,evidence_score,global_rank,n_samples,mean_sample_rank,std_sample_rank,stability_score
0,TP53,207,0.224433,0.310993,0.000240,0.037011,0.000107,0.001700,205,1.257315,1,5,63.0,9.460444,0.095598
1,MYC,217,0.226426,0.319816,0.011175,0.040126,0.000155,0.003010,214,1.246607,2,5,67.4,24.744696,0.038843
2,JUN,138,0.225626,0.300653,-0.001529,0.055776,0.000084,0.004178,136,1.176677,3,5,48.8,8.526429,0.104971
3,EGR1,92,0.225958,0.303583,0.004390,0.039504,0.000133,0.002130,92,1.062588,4,5,61.4,13.164346,0.070600
4,CEBPB,88,0.216195,0.311312,0.002128,0.057665,0.000118,0.004518,87,1.015938,5,5,90.2,10.848963,0.084396
5,NR3C1,72,0.216033,0.311654,0.001558,0.060118,0.000069,0.004497,69,1.013164,6,5,79.6,4.669047,0.176396
6,FOS,72,0.226205,0.309000,0.008814,0.041542,0.000128,0.002549,72,1.012266,7,5,59.8,11.388591,0.080719
7,STAT1,63,0.222754,0.311089,0.003791,0.045108,0.000091,0.002535,62,0.990077,8,5,59.8,6.572671,0.132054
8,HNF4A,55,0.231927,0.306210,0.013359,0.052825,0.000754,0.002881,55,0.966048,9,5,55.6,14.724130,0.063597
9,ETS1,58,0.226678,0.312364,0.003817,0.057794,0.000110,0.004194,57,0.964236,10,5,50.6,11.865918,0.077725


,driver,target,score,evidence_score,internal,external,internal_support,external_support,qvalue,n_samples,consistency_pos
0,ABL1,TP53,0.318607,0.408831,0.044148,0.025215,4.098203e-04,1.257855e-04,0.007444,5,1.0
1,ABL1,BCL6,0.286138,0.374736,-0.000986,0.017291,1.190018e-05,3.029899e-05,0.007195,5,1.0
2,ABL1,JUN,0.275005,0.356806,0.009014,0.022921,2.104661e-05,2.175701e-04,0.006449,5,1.0
3,ABL1,BAX,0.231585,0.300107,-0.000411,0.041199,1.973293e-07,2.483119e-04,0.005278,5,1.0
4,AHR,MYC,0.283034,0.365567,-0.041479,0.051172,5.484404e-05,3.325345e-04,0.005278,5,1.0
5,AHR,IL6,0.278900,0.361695,0.010855,0.015541,1.898266e-05,8.488333e-05,0.009956,5,1.0
6,AHR,MT2A,0.279334,0.360789,0.008708,0.098460,8.474924e-05,2.807842e-03,0.006924,5,1.0
7,AHR,CCND1,0.269858,0.351807,0.000045,0.013728,8.538740e-06,1.473878e-04,0.010849,5,1.0
8,AHR,FGFR1,0.259966,0.342003,-0.022946,-0.003556,1.514618e-05,2.635612e-07,0.005896,5,1.0
9,AHR,MMP9,0.254148,0.338932,0.001103,-0.010343,3.140827e-05,0.000000e+00,0.006195,5,1.0


,source,target,n_samples,score,std_score,bio_mean_score,bio_std_score,consistency_pos,pvalue_effect,pvalue_sign,...,external_y,internal_support_y,external_support_y,full_r2_y,attention_weight,cascade_gain,internal_purity_y,external_confounding_penalty_y,intrinsic_score_y,external_branch_score_y
0,MYC,CDK4,5,0.319816,0.019372,0.037903,0.017659,1.0,0.004326,0.03125,...,0.033865,1.766087e-04,3.626123e-05,0.045804,0.729183,1.217773,0.755461,3.194970e-05,0.165387,3.626123e-05
1,YBX1,TYMS,5,0.312533,0.025980,0.040400,0.018447,1.0,0.004030,0.03125,...,0.057595,6.605950e-05,1.767849e-04,0.053594,0.693478,1.218209,0.393418,1.603753e-04,0.144786,1.767849e-04
2,CDX2,UGT2B17,5,0.312424,0.048379,0.043846,0.063921,1.0,0.099930,0.03125,...,0.080717,2.563848e-08,5.190465e-04,0.015146,0.646558,1.204887,0.002119,5.151730e-04,0.113758,5.190465e-04
3,ETS1,CTSB,5,0.312364,0.009665,0.030075,0.017629,1.0,0.009432,0.03125,...,0.038041,7.797879e-06,4.970653e-05,0.067140,0.678026,1.213339,0.277116,4.736401e-05,0.131193,4.970653e-05
4,NR3C1,ATF2,5,0.311654,0.012286,0.028476,0.019230,1.0,0.014810,0.03125,...,-0.005825,3.544555e-05,0.000000e+00,0.026584,0.743636,1.201777,0.970334,0.000000e+00,0.165699,0.000000e+00
5,STAT1,MMP9,5,0.311089,0.026866,0.030279,0.032994,1.0,0.054715,0.03125,...,0.021368,1.443441e-07,9.713573e-06,0.003408,0.655421,1.209080,0.083293,9.588415e-06,0.116547,9.713573e-06
6,TP53,PTPA,5,0.310993,0.010298,0.036061,0.016145,1.0,0.003760,0.03125,...,0.012563,1.561678e-04,1.898819e-06,0.028643,0.745383,1.209061,0.950166,1.690784e-06,0.171108,1.898819e-06
7,JUND,IL6,5,0.309396,0.027092,0.036979,0.017494,1.0,0.004563,0.03125,...,0.017119,8.069788e-05,4.986424e-06,0.004316,0.736731,1.200702,0.869897,4.545342e-06,0.161985,4.986424e-06
8,TP53,HSP90AB1,5,0.307547,0.021371,0.036737,0.016888,1.0,0.004127,0.03125,...,0.071664,6.185054e-05,2.951245e-04,0.141322,0.685526,1.225080,0.308150,2.668541e-04,0.142000,2.951245e-04
9,YBX1,TOP2A,5,0.306902,0.020836,0.036600,0.011654,1.0,0.001083,0.03125,...,-0.008807,1.784466e-04,0.000000e+00,0.036669,0.748040,1.211249,0.976885,0.000000e+00,0.175542,0.000000e+00


,driver,mechanism_axis,n_overlap,overlap_genes
0,ETV4,invasion_front_local_invasion,6,CXCR4;ETV4;MMP9;PLAU;PLAUR;VIM
1,SNAI2,invasion_front_local_invasion,5,CXCL12;CXCR4;KLF4;MMP9;VIM
2,ATF2,invasion_front_local_invasion,3,FN1;MMP9;PLAU
3,ETS1,invasion_front_local_invasion,3,CXCR4;MMP9;PLAU
4,FOS,invasion_front_local_invasion,3,MMP9;PLAU;PLAUR
5,JUN,invasion_front_local_invasion,3,MMP9;PLAU;PLAUR
6,KLF5,invasion_front_local_invasion,3,CXCR4;KLF4;MMP9
7,SNAI2,epithelial_stem_like_state,3,CLDN1;KRT8;SOX9
8,AHR,hypoxia_regulon,2,SLC2A1;VEGFA
9,APEX1,ccc_cxcl_immune_recruitment_axis,2,CCL2;CXCL8


,driver,best_niche,best_niche_score,second_best_niche_score,specificity_margin
0,SATB2,P5CRC:niche_1,0.791432,0.679716,0.111716
1,TGIF1,P5CRC:niche_0,0.657536,0.550883,0.106653
2,PBX1,P5CRC:niche_1,0.715682,0.617487,0.098195
3,NFKBIZ,P5CRC:niche_1,0.810277,0.712283,0.097993
4,TP53,P5CRC:niche_1,0.785302,0.705465,0.079837
5,NFIX,P5CRC:niche_1,0.650405,0.572120,0.078285
6,NFIB,P5CRC:niche_1,0.619871,0.543216,0.076655
7,CDX2,P5CRC:niche_1,0.748714,0.673300,0.075414
8,KAT6B,P5CRC:niche_1,0.665487,0.595489,0.069998
9,NR1I2,P5CRC:niche_1,0.738817,0.670243,0.068575


,gene,variance,nonzero_fraction,keep_after_qc
1206,MT-ATP6,6.763826,0.616000,True
1381,PIGR,4.076301,0.212500,True
1887,VIM,4.015887,0.276000,True
680,FTH1,3.874358,0.377500,True
387,COL3A1,3.606773,0.214667,True
1732,TAGLN,3.442872,0.180333,True
135,B2M,2.912144,0.307167,True
1043,JCHAIN,2.738782,0.132500,True
377,COL1A1,2.702720,0.144333,True
1225,MYL9,2.697265,0.144333,True


results saved to: C:\Users\Simon\Downloads\crc_grn_optimized_v3\crc_grn_optimized_v3\results\real_run
